# 01 · Suppression-mechanism confirmation  —  `DIFF@pivot = ATTN(q = −PROGRESS, k = TASK)`

Hypothesis: at the pivot the goal heads compute the leftover task by a single attention op whose
**query is a function of PROGRESS** (the done-set) and whose **keys/values are the TASK spec**. The
query *suppresses* attention onto the task items already satisfied by PROGRESS; the surviving mass is
the frontier item, written by the OV.

Battery (one 3-predicate example, 2–3 instances — flow-polishing only, no aggregate stats):
- **A1** graded suppression + OV-write norm — does attention onto item-k collapse exactly when PROGRESS completes k?
- **A2** anti-alignment — is the query update from completing item i anti-parallel to item-i's task keys?
- **A3** sign necessity — reflecting the done-update through baseline should *not* skip to the frontier.
- **B1** query linearity (geometric) — `Δq({a,b}) ≈ Δq({a}) + Δq({b})`?
- **B2** compositional causal injection — inject `q({a})+q({b})−q({})`; does it *generate* item c only? saturation → boundary.

### 🔧 Intervention mechanism (reference) — pre-RoPE query transplant

*The same machinery underlies Spec 1 (`00_diff.ipynb`), A3, and B2. Read this once; every
`inject`-style arm below is a variation on it.* Numbers below are for **Qwen2.5-Coder-32B**
(40 q-heads, 8 kv-heads, `head_dim=128` → `q_proj` output width = 40×128 = 5120, `GROUP = 40/8 = 5`).

#### 1 · Source / destination prompts + pivots

Both prompts are the **same TASK**, differing only in how much of the prefill (**PROGRESS**) is
written. The task description precedes the prefill, so its tokens sit at **identical positions** in both
→ their **keys/values match and are never touched**. Only the **pivot query** differs.

```
                    ┌────────────── SHARED: TASK spec  (KEYS / VALUES live here) ──────────────┐
 SOURCE (pab)       │ Write func(words) that computes:                                          │
 "a,b done"         │  (1) `a`: words longer than 3 chars                                        │
                    │  (2) `b`: count of palindromic words                                       │
                    │  (3) `c`: length of the longest word                                       │
                    └───────────────────────────────────────────────────────────────────────────┘
                    <code>
                    def func(words: list[str]):
                        a = [w for w in words if len(w) > 3]   ┐ PROGRESS = {a,b}
                        b = sum(w == w[::-1] for w in words)   ┘ (this region DIFFERS)
                        ‸   ← PIVOT_SRC   (trailing indent = last token; about to write next var)

 DEST (p0)          ┌────────────── SAME TASK, SAME TOKEN POSITIONS ───────────────────────────┐
 "nothing done"     │ ... identical (1)(2)(3) ...                                                │
                    └───────────────────────────────────────────────────────────────────────────┘
                    <code>
                    def func(words: list[str]):
                        ‸   ← PIVOT_DEST   PROGRESS = {}
```

- **Pivot = last token of the prompt** (`pivot_of(p) = input_ids.shape[1] - 1`).
- SRC's pivot has a *larger absolute index* (extra a,b lines) — harmless: the query is cached
  **pre-RoPE** (position-independent) and re-rotated at DEST's own position on injection.

#### 2 · What is cached from the SRC pivot

```python
qab = cache_src_proj(pab, [pivot_of(pab)], Q_LAYERS, "q")   # {layer: (1, 5120)}
```

- Traces each goal-layer's `…self_attn.q_proj` **output** at the **pivot position only**.
- It is the **`q_proj` output, PRE-RoPE** (RoPE is applied later, inside attention).
- The full 5120-wide row is stored, but **only the goal-head 128-wide slices are used** on injection.
- `q_reflect` has no source prompt — it is *synthesized*: `q_lincomb([(2.0, q0), (-1.0, qab)])` = `2·q0 − qab`.

#### 3 · The injection — overwrite only goal-head q-slices at the pivot

`proj_patch_hooks` registers a forward hook on each `q_proj`; for each goal head `h` and target
position `p` it does `o[:, p, h*128:(h+1)*128] = qab[layer][row, h*128:(h+1)*128]`. All other heads
keep DEST's own query; **keys/values (TASK) are untouched**. RoPE rotates the injected vector at `p`.

```
   DEST q_proj output @ PIVOT_DEST, layer L:   [ h0 | h1 |███| h3 | … |███| … | h39 ]
                                                          ▲            ▲
                                                    goal heads overwritten with qab's slices
                                                    (███), everything else = DEST's own query
```

**Two phases (`greedy`):**

```
   PREFILL  ...task...  def func(...):  [PIVOT_DEST]‸    ← prefill_specs (positions=[PIVOT_DEST])
            → the pivot's patched attention leaves a modified residual + K/V in the KV cache
              (this one-shot footprint "rides the cache"; the QUERY itself is never cached)

   DECODE 1     [tok₁]‸    ← decode_specs (positions=[-1]): patch the NEW token's own query
   DECODE 2     [tok₂]‸    ← same fixed qab again  (queries are recomputed fresh every step)
   ...                        repeated for the first n_steer steps, then released
```

- **Queries are never stored in the KV cache — only K and V are.** So the prefill patch's persistence
  is only the pivot's *downstream K/V footprint*; to actually **hold** the steer you must re-inject the
  query every decode step (`decode_specs`, `positions=[-1]`).
- Each step stamps the **same fixed pre-RoPE `qab`** onto an ordinary (non-pivot) generated token,
  overwriting the query it would have computed. This **freezes PROGRESS** → run forever it would loop;
  `n_steer` caps it (e.g. 20) so the frontier decision carries through without runaway.

> **vs Spec 1 (`00_diff`):** identical mechanism (pre-RoPE `q_proj`-output transplant, re-clamped per
> step). Differences are incidental: head set (`UNION_HEADS` there vs `GOAL_HEADS` here), readout
> (attention-delta/goalscope vs greedy generation), and A3's synthetic `q_reflect` control.


In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0, 2, 3")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("..")  # so `_diff_primitives` is importable
import torch, numpy as np
import _diff_primitives as P

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct")  # loads model + head masks (once)
from _diff_primitives import *  # helper functions into scope

mt = P.mt
GOAL_HEADS, BOUNDARY_HEADS, UNION_HEADS, GROUP = (
    P.GOAL_HEADS,
    P.BOUNDARY_HEADS,
    P.UNION_HEADS,
    P.GROUP,
)

2026-09-01 16:49:14 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


In [2]:
# ── intervention head set: which heads receive the pre-RoPE query transplant (A3 · B2 · B2b) ──
# Flip this to change the causal payload. Options: GOAL_HEADS · BOUNDARY_HEADS · UNION_HEADS
# (= goal ∪ boundary). Custom compose: sorted(set(GOAL_HEADS) | set(BOUNDARY_HEADS)).
# NOTE: readouts A1/A2/B1 still use GOAL_HEADS directly — this global governs interventions only.
PATCH_HEADS = GOAL_HEADS
# PATCH_HEADS = sorted(UNION_HEADS)  # goal ∪ boundary
print(f"PATCH_HEADS = {len(PATCH_HEADS)} heads over layers {sorted({l for l, _ in PATCH_HEADS})}")


PATCH_HEADS = 122 heads over layers [28, 29, 30, 31, 32, 34, 35, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55]


In [3]:
# ── multipredicate example: 4 INDEPENDENT computations, one assignment line each ──
# All orderings valid — the pivot choice is a genuine "which sub-task next", not a data dependency.
# Item `a` is the ANCHOR: ALWAYS prefilled — one in-context example that fixes the coding style
# (an inline list-comprehension). Progress analysis focuses on the remaining {b, c, d}.

USE_NATURAL_NAMES = True        # False → slot names a/b/c/d ; True → descriptive variable names
ITEMS_CONFIG      = "words_D"    # ← pick a preset from ITEMS_PRESETS below

# each preset: slot -> (condition text, RHS expression, natural variable name).
# convention: slot `a` is always a list-comprehension filter (the anchor / style example).
# all presets share the domain  def func(words: list[str])  so build_prefill/QUESTION stay valid.
ITEMS_PRESETS = {
    "words_A": {
        "a": ("the list of words that start with 'a'",          "[w for w in words if w.startswith('a')]",           "a_words"),
        "b": ("the count of palindromic words",                 "sum(w == w[::-1] for w in words)",                  "palindrome_count"),
        "c": ("the length of the longest word",                 "max(len(w) for w in words)",                        "longest_len"),
        "d": ("the list of words with more than five letters",  "[w for w in words if len(w) > 5]",                  "long_words"),
    },
    "words_B": {
        "a": ("the list of words that contain the letter 'e'",  "[w for w in words if 'e' in w]",                    "words_with_e"),
        "b": ("the total number of characters across all words","sum(len(w) for w in words)",                        "total_chars"),
        "c": ("the number of distinct words",                   "len(set(words))",                                   "distinct_count"),
        "d": ("the list of words sorted by length",             "sorted(words, key=len)",                            "by_length"),
    },
    "words_C": {
        "a": ("the list of words that end with 's'",            "[w for w in words if w.endswith('s')]",             "plural_words"),
        "d": ("the average word length",                        "sum(len(w) for w in words) / len(words)",           "avg_len"),
        "c": ("the number of words containing a digit",         "sum(any(ch.isdigit() for ch in w) for w in words)", "digit_word_count"),
        "b": ("the list of capitalized words",                  "[w.capitalize() for w in words]",                   "capitalized"),
    },
    "words_D": {
        "a": ("the list of words that end with 's'",            "[w for w in words if w.endswith('s')]",             "plural_words"),
        "b": ("the number of words with all unique characters", "sum(1 for w in words if len(w) == len(set(w)))",    "unique_chars_count"),
        "c": ("the number of words containing a digit",         "sum(any(ch.isdigit() for ch in w) for w in words)", "digit_word_count"),
        "d": ("the list of words that are palindromes",         "[w for w in words if w == w[::-1]]",                "palindromes"),
    },
}
ITEMS   = ITEMS_PRESETS[ITEMS_CONFIG]
ORDER   = ["a", "b", "c", "d"]
ANCHOR  = "a"                    # always prefilled, never a pivot choice
ANALYZE = ["b", "c", "d"]       # the progress set we sweep / inject over

def varname(k): return ITEMS[k][2] if USE_NATURAL_NAMES else k
def assign(k):  return f"{varname(k)} = {ITEMS[k][1]}"

# conditions only — no dictated variable names; the LM names what it writes at/after the pivot
TASK_SPEC = "\n".join(f"({i+1}) {ITEMS[k][0]}" for i, k in enumerate(ORDER))
QUESTION = ("Write a Python program that implements a function `func` which, given a list of "
            "strings `words`, computes:\n" + TASK_SPEC + "\n")

def build_prefill(done, extra=""):
    """Prefill = ANCHOR (always) + assignments in `done` (⊆ ANALYZE), kept in ORDER; pivot at next indent."""
    keep = [k for k in ORDER if k == ANCHOR or k in done]
    lines = ["<code>", "def func(words: list[str]):"] + [f"    {assign(k)}" for k in keep]
    return "\n".join(lines) + "\n    " + extra   # trailing indent = pivot (about to write next var)

def build_prompt(done, extra=""):
    return get_prompt(QUESTION, build_prefill(done, extra))

def pivot_of(prompt, neg_shift=-1):
    return prepare_input(prompts=prompt, tokenizer=mt)["input_ids"].shape[1] + neg_shift

# task-item token spans are identical across done-sets (task precedes prefill) — compute once.
_p0 = build_prompt([])                        # {} means "anchor a only"
ITEM_SPAN = {k: token_span(_p0, ITEMS[k][0]) for k in ORDER}
NEG_SHIFT = -1

show_prompt("done={a} (anchor only)", _p0,               ITEM_SPAN["b"], neg_shift=NEG_SHIFT)
show_prompt("done={a,b}            ", build_prompt(["b"]), ITEM_SPAN["c"], neg_shift=NEG_SHIFT)
print(f"config={ITEMS_CONFIG}  item spans:", ITEM_SPAN, " pivot({a})=", pivot_token(_p0, patch_pos=[NEG_SHIFT]))

╭─ done={a} (anchor only)  TASK span (79, 87)  ·  pivot=['    ']  ·  L=145 ───────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the number of words with all unique characters\n                                                            │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list of words that are palindromes\n                                                                    │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     plural_words = [w for w in words if w.endswith('s')]\n                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ done={a,b}              TASK span (91, 98)  ·  pivot=['    ']  ·  L=166 ───────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the number of words with all unique characters\n                                                            │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list of words that are palindromes\n                                                                    │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     plural_words = [w for w in words if w.endswith('s')]\n                                                      │
│     unique_chars_count = sum(1 for w in words if len(w) == len(set(w)))\n                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

config=words_D  item spans: {'a': (66, 76), 'b': (79, 87), 'c': (91, 98), 'd': (102, 112)}  pivot({a})= ['    ']


## A1 · Graded suppression + OV-write norm

Anchor `a` is always prefilled. Sweep PROGRESS ∈ {{a}, {a,b}, {a,b,c}}. For each: value-weighted
attention from the pivot onto each analyzed item's spec tokens (mean over GOAL heads), and the norm of
the aggregated goal-head write.
**Predict:** attention onto item-k drops sharply when k enters PROGRESS; frontier item retains mass;
write-norm decreases monotonically toward the all-done floor.

In [4]:
from src.attention import get_attention_matrices

DONE_LEVELS = [[], ["b"], ["b", "c"]]         # PROGRESS beyond the anchor: {a}, {a,b}, {a,b,c}
tbl = Table(title="A1 · pivot→TASK-item attention (mean over GOAL heads)  +  ‖goal write‖", highlight=False)
tbl.add_column("PROGRESS")
for k in ANALYZE: tbl.add_column(f"attn·{k}", justify="right")
tbl.add_column("‖write‖", justify="right")
for done in DONE_LEVELS:
    p = build_prompt(done)
    piv = pivot_of(p, neg_shift=NEG_SHIFT)
    tok = prepare_input(prompts=p, tokenizer=mt)
    info = get_attention_matrices(tok, mt, value_weighted=True)
    row = ["{" + ",".join([ANCHOR] + done) + "}"]
    for k in ANALYZE:
        a = pivot_attn(info, GOAL_HEADS, list(range(*ITEM_SPAN[k])), piv).sum()
        mark = "[green]" if k not in done else "[red]"
        row.append(f"{mark}{a:.3f}[/]")
    row.append(f"{write_norm(p, GOAL_HEADS, piv):.2f}")
    tbl.add_row(*row)
rich_print(tbl)
rich_print("[grey58]green = still-undone item (expect mass), red = done item (expect suppressed); anchor a always done[/grey58]")

 A1 · pivot→TASK-item attention (mean over GOAL  
             heads)  +  ‖goal write‖             
┏━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┓
┃ PROGRESS ┃ attn·b ┃ attn·c ┃ attn·d ┃ ‖write‖ ┃
┡━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━┩
│ {a}      │  2.883 │  0.513 │  0.325 │  227.77 │
│ {a,b}    │  0.658 │  2.294 │  0.374 │  218.27 │
│ {a,b,c}  │  0.226 │  0.273 │  2.772 │  203.09 │
└──────────┴────────┴────────┴────────┴─────────┘

green = still-undone item (expect mass), red = done item (expect suppressed); anchor a always done

## A2 · Anti-alignment of the query update with the completed item's keys

Baseline is the anchor `{a}`. `q` is pre-RoPE at the pivot; `k` is pre-RoPE at item-b's spec tokens.
For every GOAL q-head, cosine of `Δq = q({a,b}) − q({a})` (q-head slice) against the mean item-b key
(its kv-head slice, GQA `h//GROUP`).
**Predict:** cosine < 0 (the completed item is *subtracted into* the query, suppressing its keys).

In [5]:
qheads = heads_per_layer(GOAL_HEADS, "q")
# cache band must cover BOTH the readout heads (GOAL) and the intervention heads (PATCH_HEADS),
# else proj_patch_hooks hits a layer the caches lack (KeyError) when PATCH_HEADS ⊈ GOAL band.
Q_LAYERS = sorted({l for l, _ in GOAL_HEADS} | {l for l, _ in PATCH_HEADS})
p0, pb = build_prompt([]), build_prompt(["b"])       # baseline = anchor {a}; complete item b
q0 = cache_src_proj(p0, [pivot_of(p0, neg_shift=NEG_SHIFT)], Q_LAYERS, "q")
qb = cache_src_proj(pb, [pivot_of(pb, neg_shift=NEG_SHIFT)], Q_LAYERS, "q")
dq = q_lincomb([(1.0, qb), (-1.0, q0)])              # done-update from completing b: q({a,b}) − q({a})

# mean item-b KEY at its spec tokens (positions identical across prompts; use p0)
kb = cache_src_proj(p0, list(range(*ITEM_SPAN["b"])), Q_LAYERS, "k")
kbar = {l: kb[l].mean(0, keepdim=True) for l in Q_LAYERS}     # {layer: (1, width)}

hd = mt.head_dim
coss = []
for (l, h) in GOAL_HEADS:
    if l not in dq: continue
    qv = dq[l][0, h * hd:(h + 1) * hd]
    kvh = h // GROUP
    kv = kbar[l][0, kvh * hd:(kvh + 1) * hd]
    coss.append(cos(qv, kv))
coss = np.array(coss)
print(f"Δq·k(item b) cosine over {len(coss)} GOAL q-heads: "
      f"mean={coss.mean():+.3f}  median={np.median(coss):+.3f}  frac<0={ (coss < 0).mean():.2f}")
print("(strongly negative mean ⇒ completing b subtracts b's key-direction into the query — suppression)")

Δq·k(item b) cosine over 122 GOAL q-heads: mean=-0.082  median=-0.094  frac<0=0.80
(strongly negative mean ⇒ completing b subtracts b's key-direction into the query — suppression)


## A3 · Sign necessity — reflect the done-update

If the *sign* of the query update selects, then reflecting the done-update through baseline —
`q_reflect = q({a}) − (q({a,b,c}) − q({a})) = 2·q({a}) − q({a,b,c})` — should **not** behave as if b,c
are done (should not skip to d), unlike the true `q({a,b,c})`. Injected into the `{a}`-baseline dest and generated.

In [6]:
# ── A3: reflect the done-update over {b,c}; sign necessity (see reference cell) ──
pbc = build_prompt(["b", "c"])                       # {a,b,c} done  (anchor + b + c)
qbc = cache_src_proj(pbc, [pivot_of(pbc, neg_shift=NEG_SHIFT)], Q_LAYERS, "q")
q_reflect = q_lincomb([(2.0, q0), (-1.0, qbc)])      # 2·q({a}) − q({a,b,c})  = reflect the done-update

def q_inject_specs(qcache, dest_pivot):
    qh = heads_per_layer(PATCH_HEADS, "q")
    return ([dict(which="q", cache=qcache, heads=qh, positions=[dest_pivot])],
            [dict(which="q", cache=qcache, heads=qh, positions=[-1])])

dest_tok = prepare_input(prompts=p0, tokenizer=mt)   # DEST = baseline {a}
piv0 = pivot_of(p0, neg_shift=NEG_SHIFT)
show_prompt("SRC  = {a,b,c} done", pbc, ITEM_SPAN["d"], neg_shift=NEG_SHIFT)   # pivot shown in panel title
show_prompt("DEST = {a} (anchor)", p0,  ITEM_SPAN["b"], neg_shift=NEG_SHIFT)

# ── goalscope + ATTN_DIFF debug (B2 format): TASK span is shared across arms, so cache the clean
#    value-weighted pivot→TASK attention over PATCH_HEADS once; each arm shows patched − clean. ──
from src.attention import get_attention_matrices
_tlo, _thi = token_span(p0, TASK_SPEC)
TASK_POS   = list(range(_tlo, _thi))
task_toks  = [mt.tokenizer.decode([int(dest_tok["input_ids"][0][j])]) for j in TASK_POS]
_clean_info = get_attention_matrices(dest_tok, mt, value_weighted=True)
a_clean    = pivot_attn(_clean_info, PATCH_HEADS, TASK_POS, piv0)
show_delta_tokens(task_toks, a_clean, "CLEAN_ATTN · pivot→TASK (clean DEST · PATCH_HEADS)")
show_delta_tokens(task_toks, pivot_attn(_clean_info, GOAL_HEADS, TASK_POS, piv0),
                  "CLEAN_ATTN · pivot→TASK (clean DEST · GOAL_HEADS)")

def gen_inject(qcache, tag, color):
    pf, dc = q_inject_specs(qcache, piv0)
    iface  = make_patch_interface(mt, pf)
    rich_print(f"[{color}]{tag}:[/{color}] "
               f"\n{escape(greedy(dest_tok, prefill_specs=pf, decode_specs=dc, n_steer=32, max_new=128))}")
    # goalscope readout of the DEST under the SAME query patch (1B format; single-forward, prefill only)
    goalscope_readout(f"   ↳ goalscope · {tag}", p0, piv0, heads=PATCH_HEADS, iface=iface)
    # ATTN_DIFF (1A): value-weighted pivot→TASK attention, patched − clean, over PATCH_HEADS (heatmap only)
    _info_p = get_attention_matrices(dest_tok, mt, value_weighted=True, patch_interface=iface)
    a_p = pivot_attn(_info_p, PATCH_HEADS, TASK_POS, piv0)
    show_delta_tokens(task_toks, a_p - a_clean, f"ATTN_DIFF · {tag} (patched − clean · PATCH_HEADS)")
    # actual (non-delta) value-weighted pivot→TASK attention over GOAL_HEADS — where goal heads look
    show_delta_tokens(task_toks, pivot_attn(_info_p, GOAL_HEADS, TASK_POS, piv0),
                      f"ATTN · {tag} (patched · value-weighted · GOAL_HEADS)")

# goalscope readouts — SRC (q(b,c) at its own pivot) + clean DEST, then patched arms fold into gen_inject
goalscope_readout("SRC   q(b,c) = {a,b,c}", pbc, pivot_of(pbc, neg_shift=NEG_SHIFT), heads=PATCH_HEADS)
goalscope_readout("clean DEST   = {a}    ", p0,  piv0,          heads=PATCH_HEADS)

rich_print(f"[grey70]clean {{a}} (expect writes b) :[/grey70] \n{escape(greedy(dest_tok, max_new=128))}")
gen_inject(qbc,       "inject q(b,c) (expect d)  ", "sea_green3")
gen_inject(q_reflect, "inject q_reflect (expect ≠d)", "salmon1")

╭─ SRC  = {a,b,c} done  TASK span (102, 112)  ·  pivot=['    ']  ·  L=186 ────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the number of words with all unique characters\n                                                            │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list of words that are palindromes\n                                                                    │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     plural_words = [w for w in words if w.endswith('s')]\n                                                      │
│     unique_chars_count = sum(1 for w in words if len(w) == len(set(w)))\n                                       │
│     digit_word_count = sum(any(ch.isdigit() for ch in w) for w in words)\n                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DEST = {a} (anchor)  TASK span (79, 87)  ·  pivot=['    ']  ·  L=145 ──────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the number of words with all unique characters\n                                                            │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list of words that are palindromes\n                                                                    │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     plural_words = [w for w in words if w.endswith('s')]\n                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

CLEAN_ATTN · pivot→TASK (clean DEST · PATCH_HEADS)   (green +Δ / red −Δ, |Δ|max=1.172)

CLEAN_ATTN · pivot→TASK (clean DEST · GOAL_HEADS)   (green +Δ / red −Δ, |Δ|max=1.172)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


... (10 outputs left out)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · inject q_reflect (expect ≠d)  pos=144 '    ' => I'm checking if all characters in a string are 
unique by comparing the length of the string to the length of a set created from the string, which inherently 
contains only unique elements.

ATTN_DIFF · inject q_reflect (expect ≠d) (patched − clean · PATCH_HEADS)   (green +Δ / red −Δ, |Δ|max=1.345)

ATTN · inject q_reflect (expect ≠d) (patched · value-weighted · GOAL_HEADS)   (green +Δ / red −Δ, |Δ|max=2.517)

In [7]:
# rich_print(f"[sea_green3]inject q(b,c) (expect d)   :[/sea_green3] "
#            f"\n{escape(greedy(dest_tok, prefill_specs=pf_bc, decode_specs=dc_bc, n_steer=3, max_new=128))}")
# rich_print(f"[salmon1]inject q_reflect (expect ≠d) :[/salmon1] "
#            f"\n{escape(greedy(dest_tok, prefill_specs=pf_rf, decode_specs=dc_rf, n_steer=3, max_new=128))}")

## A4 · Cross-TASK KEY+VALUE transplant (Spec-2 mechanism) — is the *next goal* read from the task k/v?

SRC and DST are two 4-condition Python tasks that **share conditions 1 & 2** (so `SRC.PROGRESS == DST.PROGRESS`, both prefilled in) and **differ at conditions 3 & 4**. Both prompts sit at the same pivot: conditions 1 & 2 done, about to write the 3rd.

Transplant SRC's **key + value** over the divergent **tail span (conditions 3 & 4)** onto DST at the pivot (right-edge aligned, GQA-corrected per-query-head routing — `GQA_BLEED=False` **always**; the bled path was debug-only). PROGRESS and pivot position are held constant, so the *only* variable is the task-tail content.

**Predict:** clean DST writes D3; **patched DST writes S3** (the source's 3rd condition) — i.e. the "what to do next" is read from the task-description k/v that the pivot attends to. Success is judged qualitatively from the generations + goalscope readouts.

In [47]:
# ── A4 setup: two 4-condition Python tasks sharing conditions 1&2, differing at 3&4 ──
ALIGN_KNOB = "right"     # "right" = right-edge align (later tokens carry more context) ;
                       # "left"  = share the tail START (conds 1&2 identical => left-anchored) ;
                       # "pad"   = left-pad the SHORTER tail so both tails span the SAME positions,
                       #           fully overwriting the longer (dest) tail (no dest-tail residue).
# Each condition: (description, inline RHS expression, natural variable name). Progress var
# names honor USE_NATURAL_NAMES: slot letters a/b/c/d by position when False (so SRC & DST
# share the pivot name 'c' — isolating body from name), else the descriptive name.
USE_NATURAL_NAMES = True        # False → slot names a/b/c/d ; True → descriptive variable names

A4_SHARED = [
    ("the list of words that end with 's'", "[w for w in words if w.endswith('s')]", "plural_words"),
    ("the list of words that are palindromes", "[w for w in words if w == w[::-1]]",       "palindromes"),
]
A4_SRC_TAIL = [   # conditions 3 & 4 for SRC
    ("the number of distinct words",        "len(set(words))",                        "distinct_count"),
    ("the average word length",                "sum(len(w) for w in words) / len(words)",  "avg_len"),
]
A4_DST_TAIL = [   # conditions 3 & 4 for DST
    ("the number of words containing a digit", "sum(any(c.isdigit() for c in w) for w in words)", "digit_words"),
    ("the list of capitalized words",          "[w.capitalize() for w in words]",                 "capitalized"),
]

A4_SRC_TAIL, A4_DST_TAIL = A4_DST_TAIL, A4_SRC_TAIL

A4_QUESTION = ("Write a Python program that implements a function `func` which, given a list of "
               "strings `words`, computes:\n{task}\n")
def _a4_name(i, cond):   # position i (0-based over the full 4-cond list) -> var name
    return cond[2] if USE_NATURAL_NAMES else chr(ord("a") + i)
def a4_build(tail, pad_n=0):
    # pad_n pad tokens inserted right before the tail "(3)" (ALIGN_KNOB=="pad") — length-matches
    # the two tails so the transplant covers the longer tail exactly.  pad = tokenizer.pad_token
    # (<|endoftext|>, atomic 1-token); <|fim_pad|> is a more-neutral alt if EOS-semantics bleed.
    shared  = "\n".join(f"({i+1}) {c[0]}" for i, c in enumerate(A4_SHARED))
    tailtxt = "\n".join(f"({i+1+len(A4_SHARED)}) {c[0]}" for i, c in enumerate(tail))
    task    = shared + "\n" + mt.tokenizer.pad_token * pad_n + tailtxt
    prog    = "\n".join(f"    {_a4_name(i, c)} = {c[1]}" for i, c in enumerate(A4_SHARED))  # conds 1&2 implemented
    prefill = "<code>\ndef func(words: list[str]):\n" + prog + "\n    "  # trailing indent = pivot (write 3rd)
    return get_prompt(A4_QUESTION.format(task=task), prefill)

a4_src_prompt = a4_build(A4_SRC_TAIL)
a4_dst_prompt = a4_build(A4_DST_TAIL)

# divergent TAIL span = conditions 3&4 (the ONLY differing text); right-edge align like Spec-2
def _a4_tail_str(tail): return "\n".join(f"({i+3}) {c[0]}" for i, c in enumerate(tail))
A4_SRC_SPAN = token_span(a4_src_prompt, _a4_tail_str(A4_SRC_TAIL))
A4_DST_SPAN = token_span(a4_dst_prompt, _a4_tail_str(A4_DST_TAIL))
_len_s, _len_d = A4_SRC_SPAN[1] - A4_SRC_SPAN[0], A4_DST_SPAN[1] - A4_DST_SPAN[0]
if ALIGN_KNOB == "pad":                                              # length-match via left-pad
    assert A4_SRC_SPAN[0] == A4_DST_SPAN[0], "tails must share a start (conds 1&2 identical)"
    _tail0, A4_K = A4_SRC_SPAN[0], max(_len_s, _len_d)               # common start; padded length
    if   _len_s < _len_d: a4_src_prompt = a4_build(A4_SRC_TAIL, pad_n=_len_d - _len_s)  # pad SRC
    elif _len_d < _len_s: a4_dst_prompt = a4_build(A4_DST_TAIL, pad_n=_len_s - _len_d)  # pad DST
    A4_SRC_POS  = list(range(_tail0, _tail0 + A4_K))                 # SRC tail incl. pads
    A4_DST_POS  = list(range(_tail0, _tail0 + A4_K))                 # DST tail (fully covered)
    A4_SRC_SPAN = A4_DST_SPAN = (_tail0, _tail0 + A4_K)              # for the viz panels below
elif ALIGN_KNOB == "right":                                          # right-edge
    A4_K = min(_len_s, _len_d)
    A4_SRC_POS = list(range(A4_SRC_SPAN[1] - A4_K, A4_SRC_SPAN[1]))  # last K of each tail
    A4_DST_POS = list(range(A4_DST_SPAN[1] - A4_K, A4_DST_SPAN[1]))
elif ALIGN_KNOB == "left":                                           # left-edge (shared start)
    A4_K = min(_len_s, _len_d)
    A4_SRC_POS = list(range(A4_SRC_SPAN[0], A4_SRC_SPAN[0] + A4_K))  # first K of each tail
    A4_DST_POS = list(range(A4_DST_SPAN[0], A4_DST_SPAN[0] + A4_K))
else:
    raise ValueError(f"ALIGN_KNOB must be 'left'|'right'|'pad', got {ALIGN_KNOB!r}")
A4_PIV_SRC  = pivot_of(a4_src_prompt)   # last token (write 3rd) — k/v transplant reads at -1, not NEG_SHIFT
A4_PIV_DST  = pivot_of(a4_dst_prompt)

# ── mark the EXACT positions read (SRC) / overwritten (DST) — yellow = the transplanted tokens,
#    salmon = pivot. NOTE alignment: the shared prefix (conds 1&2) is identical, so both tail spans
#    START at the same index; right-edge (Spec-2) alignment therefore drops the FRONT of the longer
#    tail when the two tails differ in length. This panel exposes exactly which tokens move. ──
_ids_s = prepare_input(prompts=a4_src_prompt, tokenizer=mt)["input_ids"][0].tolist()
_ids_d = prepare_input(prompts=a4_dst_prompt, tokenizer=mt)["input_ids"][0].tolist()
rich_print(Panel(Text.from_markup(exact_prompt(a4_src_prompt, patch_pos=[A4_PIV_SRC], emphasize_tokens=A4_SRC_POS)),
                 title=f"[bold]A4 SRC · READ span A4_SRC_POS={A4_SRC_POS} (yellow) · pivot salmon[/bold]",
                 title_align="left", border_style="grey42", padding=(1, 1)))
rich_print(Panel(Text.from_markup(exact_prompt(a4_dst_prompt, patch_pos=[A4_PIV_DST], emphasize_tokens=A4_DST_POS)),
                 title=f"[bold]A4 DST · REPLACED span A4_DST_POS={A4_DST_POS} (yellow) · pivot salmon[/bold]",
                 title_align="left", border_style="grey42", padding=(1, 1)))
_len_s, _len_d = A4_SRC_SPAN[1] - A4_SRC_SPAN[0], A4_DST_SPAN[1] - A4_DST_SPAN[0]
print(f"tail spans: SRC {A4_SRC_SPAN} (len {_len_s})  DST {A4_DST_SPAN} (len {_len_d})  K={A4_K}  "
      f"| heads=PATCH_HEADS (n={len(PATCH_HEADS)})")
print("SRC read     :", repr("".join(mt.tokenizer.decode([_ids_s[j]]) for j in A4_SRC_POS)))
print("DST replaced :", repr("".join(mt.tokenizer.decode([_ids_d[j]]) for j in A4_DST_POS)))
print("DST kept (un-replaced tail):",
      repr("".join(mt.tokenizer.decode([_ids_d[j]]) for j in range(A4_DST_SPAN[0], A4_DST_SPAN[1]) if j not in set(A4_DST_POS))))
if _len_s != _len_d:
    print(f"  ⚠ tail lengths differ by {abs(_len_s - _len_d)} tokens — right-edge align leaves the FRONT "
          f"of the longer tail untouched (e.g. start of D3 stays dest → leakage). Consider left-edge align.")

╭─ A4 SRC · READ span A4_SRC_POS=[93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108] (yello─╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the list of words that are palindromes\n                                                                    │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list of capitalized words\n                                                                             │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     plural_words = [w for w in words if w.endswith('s')]\n                                                      │
│     palindromes = [w for w in words if w == w[::-1]]\n                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ A4 DST · REPLACED span A4_DST_POS=[90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105] (yell─╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the list of words that are palindromes\n                                                                    │
│ (3) the number of distinct words\n                                                                              │
│ (4) the average word length\n                                                                                   │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     plural_words = [w for w in words if w.endswith('s')]\n                                                      │
│     palindromes = [w for w in words if w == w[::-1]]\n                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

tail spans: SRC (90, 109) (len 19)  DST (90, 106) (len 16)  K=16  | heads=PATCH_HEADS (n=228)
SRC read     : ' the number of words containing a digit\n(4) the list of capitalized words'
DST replaced : '(3) the number of distinct words\n(4) the average word length'
DST kept (un-replaced tail): ''
  ⚠ tail lengths differ by 3 tokens — right-edge align leaves the FRONT of the longer tail untouched (e.g. start of D3 stays dest → leakage). Consider left-edge align.


In [48]:
PATCH_HEADS = UNION_HEADS  # goal ∪ boundary
len(PATCH_HEADS)

228

In [49]:
print(a4_src_prompt)

<|im_start|>system
You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just the code, no need to comment or explain.<|im_end|>
<|im_start|>user
Write a Python program that implements a function `func` which, given a list of strings `words`, computes:
(1) the list of words that end with 's'
(2) the list of words that are palindromes
(3) the number of words containing a digit
(4) the list of capitalized words
<|im_end|>
<|im_start|>assistant
<code>
def func(words: list[str]):
    plural_words = [w for w in words if w.endswith('s')]
    palindromes = [w for w in words if w == w[::-1]]
    


In [50]:
# ── A4 intervention: transplant SRC key+value over the tail span onto DST (GQA-corrected) ──
from _diff_primitives import headwise_kv_patch, make_headwise_interface, _layers_of   # autoreload won't re-bind

a4_kvheads = heads_per_layer(PATCH_HEADS, "k", layers=set(_layers_of(PATCH_HEADS)))
a4_k = cache_src_proj(a4_src_prompt, A4_SRC_POS, sorted(a4_kvheads), "k")
a4_v = cache_src_proj(a4_src_prompt, A4_SRC_POS, sorted(a4_kvheads), "v")

a4_src_tok = prepare_input(a4_src_prompt, tokenizer=mt)
a4_dst_tok = prepare_input(a4_dst_prompt, tokenizer=mt)

# GQA_BLEED=False ALWAYS — per-query-head routing (only PATCH_HEADS read the source span)
with headwise_kv_patch(mt, PATCH_HEADS, a4_k, a4_v, A4_DST_POS):
    a4_patched = greedy(a4_dst_tok, max_new=128)

rich_print(f"[sea_green3]clean SRC    (expect S3 = palindromes) :[/sea_green3] \n{escape(greedy(a4_src_tok, max_new=128))}")
rich_print(f"[grey70]clean DEST   (expect D3 = digit-count):[/grey70] \n{escape(greedy(a4_dst_tok, max_new=128))}")
rich_print(f"[salmon1]patched DEST (expect S3 = palindromes):[/salmon1] \n{escape(a4_patched)}")

# goalscope readouts at each pivot (Spec-2 format); patched via the headwise interface
_a4_iface = make_headwise_interface(mt, PATCH_HEADS, a4_k, a4_v, A4_DST_POS)
goalscope_readout("clean SRC    ", a4_src_prompt, A4_PIV_SRC, heads=PATCH_HEADS)
goalscope_readout("clean DEST   ", a4_dst_prompt, A4_PIV_DST, heads=PATCH_HEADS)
goalscope_readout("patched DEST ", a4_dst_prompt, A4_PIV_DST, heads=PATCH_HEADS, iface=_a4_iface)

clean SRC    (expect S3 = palindromes) : 
 contains_digit = sum(any(char.isdigit() for char in w) for w in words)
    capitalized_words = [w for w in words if w.istitle()]
    return plural_words, palindromes, contains_digit, capitalized_words
</code>

clean DEST   (expect D3 = digit-count): 
 distinct_words = len(set(words))
    average_length = sum(len(w) for w in words) / len(words) if words else 0
    return plural_words, palindromes, distinct_words, average_length
</code>

patched DEST (expect S3 = palindromes): 
 num_distinct_words = len(set(words))
    avg_word_length = sum(len(w) for w in words) / len(words) if words else 0
    return plural_words, palindromes, num_distinct_words, avg_word_length
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

clean SRC      pos=160 '    ' => I'm checking for digits in the string.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

clean DEST     pos=157 '    ' => Identifying unique elements.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

patched DEST   pos=157 '    ' => I'm checking if the current character is a digit.

[{'pos': 157,
  'token': '    ',
  'patched_gen': "I'm checking if the current character is a digit.",
  'scoped': True}]

## B1 · Query linearity (geometric)

Is the done-set encoded additively in query space? Cosine, over the concatenated GOAL q-head slices,
between `Δq(b,c)` and `Δq(b) + Δq(c)`  (each `Δq(X) = q({a}∪X) − q({a})`, measured against the anchor baseline).
**Predict:** high cosine ⇒ subtraction is linear/compositional over done-items.

In [8]:
# ── B setup: rebuild the B-experiment prompts/queries SELF-CONTAINED (NOT pulled from A) ──
# Every query and the dest injection point use the SAME neg_shift pivot, so B is internally consistent
# (A cached at the −1 pivot; B tunes the read/inject position via NEG_SHIFT).
NEG_SHIFT = -1                                    # read/inject the progress query this many tokens off the end
PATCH_HEADS = GOAL_HEADS
hd = mt.head_dim
# cache band must cover the readout (GOAL) AND intervention (PATCH_HEADS) layers, else
# proj_patch_hooks hits a layer the caches lack (KeyError) when PATCH_HEADS ⊄ GOAL band.
Q_LAYERS = sorted({l for l, _ in GOAL_HEADS} | {l for l, _ in PATCH_HEADS})

p0  = build_prompt([])                            # baseline / DEST = anchor {a}
pbc = build_prompt(["b", "c"])                    # {a,b,c} done
piv0     = pivot_of(p0,  neg_shift=NEG_SHIFT)     # dest injection point (neg_shift pivot)
piv_bc   = pivot_of(pbc, neg_shift=NEG_SHIFT)
dest_tok = prepare_input(prompts=p0, tokenizer=mt)
q0  = cache_src_proj(p0,  [piv0],   Q_LAYERS, "q")   # baseline query  q({a})
qbc = cache_src_proj(pbc, [piv_bc], Q_LAYERS, "q")   # q({a,b,c})

def q_inject_specs(qcache, dest_pivot):
    qh = heads_per_layer(PATCH_HEADS, "q")
    return ([dict(which="q", cache=qcache, heads=qh, positions=[dest_pivot])],
            [dict(which="q", cache=qcache, heads=qh, positions=[-1])])

# print the B prompts with their pivots (dest injection point / source cache point) marked
for _tag, _prm, _pv in [(f"DEST/baseline p0 = {{a}} · inject", p0, piv0),
                        (f"SRC pbc = {{a,b,c}} · cache",       pbc, piv_bc)]:
    rich_print(Panel(Text.from_markup(exact_prompt(_prm, patch_pos=[_pv])),
                     title=f"[bold]B · {_tag} @ pivot idx {_pv} (neg_shift={NEG_SHIFT}) = "
                           f"{pivot_token(_prm, patch_pos=[_pv])!r}[/bold]",
                     title_align="left", border_style="grey42", padding=(1, 1)))

╭─ B · DEST/baseline p0 = {a} · inject @ pivot idx 134 (neg_shift=-1) = ['    '] ─────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the list of capitalized words\n                                                                             │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the average word length\n                                                                                   │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ B · SRC pbc = {a,b,c} · cache @ pivot idx 164 (neg_shift=-1) = ['    '] ───────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the list of capitalized words\n                                                                             │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the average word length\n                                                                                   │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│     b = [w.capitalize() for w in words]\n                                                                       │
│     c = sum(any(ch.isdigit() for ch in w) for w in words)\n                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [9]:
pb = build_prompt(["b"])
qb = cache_src_proj(pb, [pivot_of(pb, neg_shift=NEG_SHIFT)], Q_LAYERS, "q")

pc = build_prompt(["c"])
qc = cache_src_proj(pc, [pivot_of(pc, neg_shift=NEG_SHIFT)], Q_LAYERS, "q")

# show pb and pc with their (neg_shift) pivots — the exact position each query is cached at
for _tag, _p in [("pb = {a,b}", pb), ("pc = {a,c}", pc)]:
    _piv = pivot_of(_p, neg_shift=NEG_SHIFT)
    rich_print(Panel(Text.from_markup(exact_prompt(_p, patch_pos=[_piv])),
                     title=f"[bold]B1 · {_tag} · pivot idx {_piv} (neg_shift={NEG_SHIFT}) = "
                           f"{pivot_token(_p, patch_pos=[_piv])!r}[/bold]",
                     title_align="left", border_style="grey42", padding=(1, 1)))
    goalscope_readout(f"   ↳ goalscope · {_tag}", _p, _piv, heads=PATCH_HEADS)

def goal_q_vec(qcache):
    parts = [qcache[l][0, h * hd:(h + 1) * hd] for (l, h) in GOAL_HEADS if l in qcache]
    return torch.cat(parts)

# Δq(X) = q({a}∪X) − q({a})   (each done-update measured against the anchor baseline)
dq_b  = goal_q_vec(q_lincomb([(1.0, qb),  (-1.0, q0)]))
dq_c  = goal_q_vec(q_lincomb([(1.0, qc),  (-1.0, q0)]))
dq_bc = goal_q_vec(q_lincomb([(1.0, qbc), (-1.0, q0)]))
print(f"cos( Δq(b,c) , Δq(b)+Δq(c) )      = {cos(dq_bc, dq_b + dq_c):+.3f}")
print(f"cos( Δq(b,c) , Δq(b) )            = {cos(dq_bc, dq_b):+.3f}   (single-term baseline)")
print(f"cos( Δq(b)   , Δq(c) )            = {cos(dq_b,  dq_c):+.3f}   (are the two updates ~orthogonal?)")

╭─ B1 · pb = {a,b} · pivot idx 146 (neg_shift=-1) = ['    '] ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the list of capitalized words\n                                                                             │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the average word length\n                                                                                   │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│     b = [w.capitalize() for w in words]\n                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · pb = {a,b}  pos=146 '    ' => I'm checking if any character in the string is a digit.

╭─ B1 · pc = {a,c} · pivot idx 152 (neg_shift=-1) = ['    '] ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the list of capitalized words\n                                                                             │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the average word length\n                                                                                   │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│     c = sum(any(ch.isdigit() for ch in w) for w in words)\n                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · pc = {a,c}  pos=152 '    ' => I'm counting the number of uppercase letters in the string.

cos( Δq(b,c) , Δq(b)+Δq(c) )      = +0.670
cos( Δq(b,c) , Δq(b) )            = +0.548   (single-term baseline)
cos( Δq(b)   , Δq(c) )            = +0.411   (are the two updates ~orthogonal?)


## B2 · Compositional causal injection  (the payload)

Inject a *synthesised* query `q_synth = qb + qc − q0`  (`q0 = q({a})`, `qb = q({a,b})`, `qc = q({a,c})`)
at the `{a}`-baseline pivot and generate.
**Predict:** writes item **d** only — the composed done-set behaves as {a,b,c}. Controls: real `qbc = q({a,b,c})`
should match; single terms narrow less. **Saturation** `qb+qc+qd−2·q0` → {a,b,c,d} all done → **boundary/return**.

*Note:* injection target is the fixed-geometry `{a}`-baseline prompt; only the position-free pre-RoPE queries are combined.

In [10]:
pd = build_prompt(["d"])
qd = cache_src_proj(pd, [pivot_of(pd, neg_shift=NEG_SHIFT)], Q_LAYERS, "q")
_pivd = pivot_of(pd, neg_shift=NEG_SHIFT)
rich_print(Panel(Text.from_markup(exact_prompt(pd, patch_pos=[_pivd])),
                 title=f"[bold]B2 · SRC pd = {{a,d}} · cache @ pivot idx {_pivd} = "
                       f"{pivot_token(pd, patch_pos=[_pivd])!r}[/bold]",
                 title_align="left", border_style="grey42", padding=(1, 1)))
# norm-match the injected per-head query slices BEFORE composing (the raw sum was norm-dominated):
#   per patched head, set ||qb|| = ||qc|| = m  and  ||q0|| = ||qd|| = m,  where m = mean(||qb||,||qc||).
def _renorm(cache, target):   # rescale each PATCH_HEADS q-head slice of `cache` to target[(l,h)]
    out = {l: cache[l].clone() for l in cache}
    for (l, h) in PATCH_HEADS:
        sl = slice(h * hd, (h + 1) * hd)
        v = out[l][0, sl]
        out[l][0, sl] = v * (target[(l, h)] / v.norm().clamp_min(1e-8))
    return out
_m = {(l, h): 0.5 * (qb[l][0, h*hd:(h+1)*hd].norm() + qc[l][0, h*hd:(h+1)*hd].norm())
      for (l, h) in PATCH_HEADS}                 # per-head target norm = mean(||qb||, ||qc||)
qb_n, qc_n, q0_n, qd_n = (_renorm(c, _m) for c in (qb, qc, q0, qd))

q_synth = q_lincomb([(1.0, qb_n), (1.0, qc_n), (-1.0, q0_n)])                 # norm-matched ≈ done {a,b,c}
q_sat   = q_lincomb([(1.0, qb_n), (1.0, qc_n), (1.0, qd_n), (-2.0, q0_n)])    # norm-matched all-done

from src.attention import get_attention_matrices

# ── ATTN_DIFF baseline (1A / 00_diff): TASK span is shared across all injections, so cache the
#    clean value-weighted pivot→TASK attention over PATCH_HEADS once. ──
# full task span incl. the "(1) " numbering + inter-item newlines (not just the descriptions)
_tlo, _thi = token_span(p0, TASK_SPEC)
TASK_POS  = list(range(_tlo, _thi))
task_toks = [mt.tokenizer.decode([int(dest_tok["input_ids"][0][j])]) for j in TASK_POS]
_clean_info = get_attention_matrices(dest_tok, mt, value_weighted=True)
a_clean   = pivot_attn(_clean_info, PATCH_HEADS, TASK_POS, piv0)
show_delta_tokens(task_toks, a_clean, "CLEAN_ATTN · pivot→TASK (clean DEST · PATCH_HEADS)")
show_delta_tokens(task_toks, pivot_attn(_clean_info, GOAL_HEADS, TASK_POS, piv0),
                  "CLEAN_ATTN · pivot→TASK (clean DEST · GOAL_HEADS)")

def gen_inject(qcache, tag, color):
    pf, dc = q_inject_specs(qcache, piv0)
    iface = make_patch_interface(mt, pf)
    rich_print(f"[{color}]{tag}:[/{color}] "
               f"\n{escape(greedy(dest_tok, prefill_specs=pf, decode_specs=dc, n_steer=16, max_new=64))}")
    # goalscope readout of the DEST under the SAME query patch (1B format; single-forward, prefill only)
    goalscope_readout(f"   ↳ goalscope · {tag}", p0, piv0, heads=PATCH_HEADS, iface=iface)
    # ATTN_DIFF (1A): value-weighted pivot→TASK attention, patched − clean, over PATCH_HEADS (heatmap only)
    _info_p = get_attention_matrices(dest_tok, mt, value_weighted=True, patch_interface=iface)
    a_p = pivot_attn(_info_p, PATCH_HEADS, TASK_POS, piv0)
    show_delta_tokens(task_toks, a_p - a_clean, f"ATTN_DIFF · {tag} (patched − clean · PATCH_HEADS)")
    # actual (non-delta) value-weighted pivot→TASK attention over GOAL_HEADS — where goal heads look
    show_delta_tokens(task_toks, pivot_attn(_info_p, GOAL_HEADS, TASK_POS, piv0),
                      f"ATTN · {tag} (patched · value-weighted · GOAL_HEADS)")

# goalscope readouts — source_run + clean_run (same as 1B / 00_diff); patched runs fold into gen_inject
goalscope_readout("SRC   q(b,c) = {a,b,c}", pbc, piv_bc, heads=PATCH_HEADS)
goalscope_readout("clean DEST   = {a}    ", p0,  piv0,   heads=PATCH_HEADS)

rich_print(f"[grey70]clean {{a}}              (expect b) :[/grey70] \n{escape(greedy(dest_tok, max_new=64))}")
gen_inject(qbc,     "inject q(b,c) real (expect d) ", "sea_green3")
gen_inject(q_synth, "inject q_synth = b+c−q0 (expect d)", "gold3")
gen_inject(qb,      "inject q({b})       (expect c/d)", "grey58")
gen_inject(q_sat,   "inject q_sat = b+c+d−2q0 (→bound)", "salmon1")

╭─ B2 · SRC pd = {a,d} · cache @ pivot idx 150 = ['    '] ────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the list of capitalized words\n                                                                             │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the average word length\n                                                                                   │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│     d = sum(len(w) for w in words) / len(words)\n                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

CLEAN_ATTN · pivot→TASK (clean DEST · PATCH_HEADS)   (green +Δ / red −Δ, |Δ|max=1.600)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

SRC   q(b,c) = {a,b,c}  pos=164 '    ' => I'm calculating the average length of words in a string.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

clean DEST   = {a}      pos=134 '    ' => Identifying lines starting with a capital letter.

... (13 outputs left out)

inject q_sat = b+c+d−2q0 (→bound): 
 b = [w for w in words if any(c.isdigit() for c in w)]
    c = [w for w in words if w.istitle()]
    d = sum(len(w) for w in words) / len(words) if words else 0
    return a, c, len(b),

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · inject q_sat = b+c+d−2q0 (→bound)  pos=134 '    ' => I'm checking for digits in the string and 
counting them.

ATTN_DIFF · inject q_sat = b+c+d−2q0 (→bound) (patched − clean · PATCH_HEADS)   (green +Δ / red −Δ, |Δ|max=1.643)

## B2b · De-overlap correction (Gram–Schmidt)  —  is the shared component double-counted?

`Δb = qb−q0` and `Δc = qc−q0`  (`q0 = q({a})`) are **not orthogonal** (`cos(Δb,Δc) ≠ 0`), so the plain
synth `q0 + Δb + Δc` counts their shared component `s` **twice**. Since the real `Δbc = q({a,b,c})−q0`
is **already cached**, we settle it directly — no theory:

- **add:** `Δb + Δc`  (current B2)
- **GS :** `Δb + Δc − proj_b(Δc)`  — subtract the *shared* part (the **projection**, computed per layer), keep `−q0`.

Compare each reconstruction's `cos(·, Δbc)` and residual. **If `add` already tracks `Δbc`** (strong B1
additivity) the model double-counts nothing → GS would *over-correct*. **If `GS` is closer**, the
overlap is real and the GS-corrected query is the better payload. Then inject all three and read behavior.

In [11]:
# ── B2b: measure which reconstruction matches the real Δ(b,c), then inject the GS-corrected query ──
import torch
import torch.nn.functional as F

def _flat(d): return torch.cat([d[l].flatten() for l in Q_LAYERS])
Delta_bc = _flat(qbc) - _flat(q0)                       # ground-truth "b,c done" update (vs anchor baseline)

add_parts, gs_parts, cos_bc = [], [], []
q_synth_gs = {}                                          # {layer: (1, width)} for injection
for l in Q_LAYERS:
    db = (qb[l] - q0[l])                                 # (1, width)
    dc = (qc[l] - q0[l])
    fb, fc = db.flatten(), dc.flatten()
    proj = (fb @ fc) / (fb @ fb) * db                    # shared component of Δc along Δb (per layer)
    add_parts.append((db + dc).flatten())
    gs_parts.append((db + dc - proj).flatten())
    cos_bc.append((fb @ fc) / (fb.norm() * fc.norm()))
    q_synth_gs[l] = q0[l] + db + dc - proj               # = q0 + Δb + Δc_perp   (keeps −q0)

add_recon = torch.cat(add_parts)
gs_recon  = torch.cat(gs_parts)
rich_print(f"[grey70]mean per-layer cos(Δb, Δc) = {torch.stack(cos_bc).mean():.3f}   "
           f"(0 ⇒ orthogonal, no double-count)[/grey70]")
for name, v in [("add  Δb+Δc", add_recon), ("GS   Δb+Δc−proj_bΔc", gs_recon)]:
    rich_print(f"  {name:22s}  cos(·,Δbc)={F.cosine_similarity(v, Delta_bc, dim=0).item():+.3f}   "
               f"‖resid‖={(v - Delta_bc).norm():.2f}")

# ── behavioral read: inject add / GS / real, all via gen_inject from B2 ──
rich_print(f"[grey70]clean {{a}}                    (expect b) :[/grey70] \n{escape(greedy(dest_tok, max_new=64))}")
gen_inject(q_synth,    "inject q_synth    = b+c−q0           (expect d)", "gold3")
gen_inject(q_synth_gs, "inject q_synth_gs = b+c−q0−proj_bΔc  (expect d)", "cyan")
gen_inject(qbc,        "inject q(b,c) real                  (expect d)", "sea_green3")

mean per-layer cos(Δb, Δc) = 0.472   (0 ⇒ orthogonal, no double-count)

add  Δb+Δc              cos(·,Δbc)=+0.685   ‖resid‖=309.42

GS   Δb+Δc−proj_bΔc     cos(·,Δbc)=+0.693   ‖resid‖=247.54

clean {a}                    (expect b) : 
 b = [w for w in words if w.istitle()]
    c = sum(any(char.isdigit() for char in w) for w in words)
    d = sum(len(w) for w in words) / len(words) if words else 0
    return a, b, c, d

inject q_synth    = b+c−q0           (expect d): 
 b = [w for w in words if any(c.isdigit() for c in w)]
    c = [w for w in words if w.istitle()]
    d = sum(len(w) for w in words) / len(words) if words else 0
    return a, c, len(b),

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

... (6 outputs left out)

inject q(b,c) real                  (expect d): 
 b = [len(w) for w in words]
    c = sum(b) / len(b) if b else 0
    d = [w for w in words if w[0].isupper()]
    e = sum(any(char.isdigit() for char in w) for w in words)
    return

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · inject q(b,c) real                  (expect d)  pos=134 '    ' => Calculating the average length 
of words in a string.

ATTN_DIFF · inject q(b,c) real                  (expect d) (patched − clean · PATCH_HEADS)   (green +Δ / red −Δ, 
|Δ|max=1.573)